# **REGRESIÓN EXPERIMENTO**


In [10]:
import mlflow
import mlflow.sklearn
from pyspark.sql.functions import col, array
from pyspark.sql.types import DoubleType
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


StatementMeta(, 821ce642-c187-4906-9dde-32ed30228f9e, 12, Finished, Available, Finished, False)

## 🏗️ Lectura del Dataset Consolidado (Delta Lake)

El pipeline inicia consumiendo la tabla previamente optimizada en la capa Gold.


In [11]:
# Cargar datos consolidados desde la capa Gold
df = spark.read.table("gold.dbo.gold_features_clasificacion")

# Mostrar estadísticas de la variable objetivo
df.select("total_amount").describe().show()

# Mostrar muestra
display(df.select("id_genero", "id_categoria", "quantity", "age", "price_per_unit").limit(10))


StatementMeta(, 821ce642-c187-4906-9dde-32ed30228f9e, 13, Finished, Available, Finished, False)

+-------+-----------------+
|summary|     total_amount|
+-------+-----------------+
|  count|             1000|
|   mean|            456.0|
| stddev|559.9976315551235|
|    min|             25.0|
|    max|           2000.0|
+-------+-----------------+



SynapseWidget(Synapse.DataFrame, b625ed19-9ee7-4cf0-9786-150af627bb85)

## 📐 Preparación de Variables para Modelado de Regresión

**Objetivo**: predecir `total_amount` (valor numérico continuo) a partir de atributos
demográficos y comerciales.


In [12]:
# Seleccionar features para regresión
feature_columns = ["id_genero", "id_categoria", "quantity", "age", "price_per_unit"]
label_column = "total_amount"

df_ml = df.select(feature_columns + [label_column]) \
    .na.drop() \
    .filter(col("price_per_unit") > 0) \
    .filter(col("quantity") > 0)

print(f"✓ Registros válidos: {df_ml.count():,}")


StatementMeta(, 821ce642-c187-4906-9dde-32ed30228f9e, 14, Finished, Available, Finished, False)

✓ Registros válidos: 1,000


## 🔀 División de Datos (Train/Test Split 80/20)


In [13]:
train_df, test_df = df_ml.randomSplit([0.8, 0.2], seed=42)

train_count = train_df.count()
test_count  = test_df.count()

print(f"✓ Training set: {train_count:,} registros ({train_count / (train_count + test_count) * 100:.1f}%)")
print(f"✓ Test set:     {test_count:,}  registros ({test_count  / (train_count + test_count) * 100:.1f}%)")


StatementMeta(, 821ce642-c187-4906-9dde-32ed30228f9e, 15, Finished, Available, Finished, False)

✓ Training set: 838 registros (83.8%)
✓ Test set:     162  registros (16.2%)


## 🤖 Entrenamiento Ridge y Tracking MLflow


In [14]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Convertir a pandas para Scikit-Learn
print("Convirtiendo datos a pandas...")
train_pd = train_df.select(feature_columns + [label_column]).toPandas()
test_pd  = test_df.select(feature_columns + [label_column]).toPandas()

X_train = train_pd[feature_columns].values
y_train = train_pd[label_column].values
X_test  = test_pd[feature_columns].values
y_test  = test_pd[label_column].values

print(f"Train: {len(X_train):,} registros | Test: {len(X_test):,} registros")

# Cerrar cualquier run previo
mlflow.end_run()

with mlflow.start_run(run_name="ridge_regression_v1") as run:
    alpha    = 0.01
    max_iter = 50

    # --- Parámetros ---
    mlflow.log_param("alpha",      alpha)
    mlflow.log_param("max_iter",   max_iter)
    mlflow.log_param("features",   ", ".join(feature_columns))
    mlflow.log_param("train_size", train_count)
    mlflow.log_param("test_size",  test_count)

    # --- Entrenamiento ---
    ridge_model = Ridge(alpha=alpha, max_iter=max_iter)
    ridge_model.fit(X_train, y_train)
    print("✓ Modelo Ridge entrenado")

    # --- Predicciones y Métricas ---
    y_pred = ridge_model.predict(X_test)

    rmse = mean_squared_error(y_test, y_pred, squared=False)
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)

    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae",  mae)
    mlflow.log_metric("r2",   r2)

    print(f"\n📏 Métricas de Regresión")
    print(f"  RMSE : {rmse:.4f}")
    print(f"  MAE  : {mae:.4f}")
    print(f"  R²   : {r2:.4f}")

    # --- Artefacto del modelo ---
    mlflow.sklearn.log_model(ridge_model, "ridge_model")

    run_id_regresion = run.info.run_id
    print("run_id_regresion")
    print(f"\n🏃 MLflow Run ID: {run_id_regresion}")


StatementMeta(, 821ce642-c187-4906-9dde-32ed30228f9e, 16, Finished, Available, Finished, False)

Convirtiendo datos a pandas...
Train: 838 registros | Test: 162 registros
✓ Modelo Ridge entrenado

📏 Métricas de Regresión
  RMSE : 222.8635
  MAE  : 181.1841
  R²   : 0.8526
run_id_regresion

🏃 MLflow Run ID: 8450836c-4825-4321-8ade-fbc47004aafe


# **CLUSTERING EXPERIMENTO**


In [15]:
import mlflow
import mlflow.sklearn
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql.functions import col

# Cerrar cualquier run previo
mlflow.end_run()

gold_table      = "gold.dbo.gold_features_clasificacion"
experiment_name = "segmentacion_clientes_gold"
model_name      = "kmeans_segmentacion_clientes"

mlflow.set_experiment(experiment_name)

print("==============================================================")
print(f"📥 Capa origen        : {gold_table}")
print(f"🔬 Experimento MLflow : {experiment_name}")
print(f"📦 Modelo a registrar : {model_name}")
print("==============================================================")
print("✅ Entorno de Clustering inicializado")


StatementMeta(, 821ce642-c187-4906-9dde-32ed30228f9e, 17, Finished, Available, Finished, False)

📥 Capa origen        : gold.dbo.gold_features_clasificacion
🔬 Experimento MLflow : segmentacion_clientes_gold
📦 Modelo a registrar : kmeans_segmentacion_clientes
✅ Entorno de Clustering inicializado


## 🏗️ Carga de Datos desde la Capa Gold


In [16]:
df_gold = spark.table(gold_table)

print(f"✅ Registros totales : {df_gold.count():,}")
print(f"✅ Columnas          : {len(df_gold.columns)}")


StatementMeta(, 821ce642-c187-4906-9dde-32ed30228f9e, 18, Finished, Available, Finished, False)

✅ Registros totales : 1,000
✅ Columnas          : 11


## 📐 Preparación y Normalización de Datos para Clustering


In [17]:
feature_columns_km = ["quantity", "price_per_unit", "age"]

df_ml_spark = df_gold.select(feature_columns_km) \
    .na.drop() \
    .filter(col("quantity") > 0) \
    .filter(col("price_per_unit") > 0)

print(f"✅ Registros listos tras filtros: {df_ml_spark.count():,}")

df_pandas = df_ml_spark.toPandas()

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(df_pandas)

print(f"\n🧠 Matriz X_scaled lista para K-Means: {X_scaled.shape}")
print(f"🎯 Variables: {', '.join(feature_columns_km)}")


StatementMeta(, 821ce642-c187-4906-9dde-32ed30228f9e, 19, Finished, Available, Finished, False)

✅ Registros listos tras filtros: 1,000

🧠 Matriz X_scaled lista para K-Means: (1000, 3)
🎯 Variables: quantity, price_per_unit, age


## 🤖 Entrenamiento, Evaluación y Registro del Modelo K-Means


In [18]:
mlflow.end_run()

with mlflow.start_run(run_name="kmeans_segmentacion_definitiva") as run:
    run_id_clustering = run.info.run_id
    print("run_id_clustering")
    print(f"✅ MLflow Run ID: {run_id_clustering}")

    k_clusters = 3

    mlflow.log_param("algorithm",     "KMeans")
    mlflow.log_param("n_clusters",    k_clusters)
    mlflow.log_param("features_used", ", ".join(feature_columns_km))

    print(f"🛠️ Entrenando K-Means con {k_clusters} grupos...")
    kmeans = KMeans(n_clusters=k_clusters, init="k-means++", random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(X_scaled)

    df_pandas["Cluster_ID"] = cluster_labels

    wcss = kmeans.inertia_
    mlflow.log_metric("wcss_inertia", wcss)

    print(f"\n📊 Inercia del Modelo (WCSS): {wcss:.2f}")
    print("\n👥 Distribución de clientes por cluster:")
    print(df_pandas["Cluster_ID"].value_counts().to_string())

    # Visualización
    plt.figure(figsize=(9, 5))
    sns.scatterplot(
        x=df_pandas["price_per_unit"],
        y=df_pandas["quantity"],
        hue=df_pandas["Cluster_ID"],
        palette="bright",
        alpha=0.7
    )
    plt.title(f"Segmentación de Clientes (K={k_clusters} Grupos)", fontsize=12)
    plt.xlabel("Precio por Unidad ($)", fontsize=10)
    plt.ylabel("Cantidad de Productos", fontsize=10)
    plt.grid(True, linestyle=":", alpha=0.5)
    plt.legend(title="Grupo Asignado")

    plot_path = "segmentacion_clusters.png"
    plt.savefig(plot_path)
    mlflow.log_artifact(plot_path)
    plt.close()
    print("\n📸 Gráfico guardado como artefacto en MLflow")

    mlflow.sklearn.log_model(
        sk_model=kmeans,
        artifact_path="model",
        registered_model_name=model_name,
        input_example=X_scaled[:1]
    )
    print(f"📦 Modelo K-Means registrado como: '{model_name}'")

print("\n🏁 CLUSTERING COMPLETADO")


StatementMeta(, 821ce642-c187-4906-9dde-32ed30228f9e, 20, Finished, Available, Finished, False)

run_id_clustering
✅ MLflow Run ID: e99eafea-676b-485f-a7a6-c4e86f6854e0
🛠️ Entrenando K-Means con 3 grupos...

📊 Inercia del Modelo (WCSS): 1586.25

👥 Distribución de clientes por cluster:
Cluster_ID
2    361
0    343
1    296

📸 Gráfico guardado como artefacto en MLflow
📦 Modelo K-Means registrado como: 'kmeans_segmentacion_clientes'


2026-06-15:03:52:58,766 ERROR    [shared_platform_utils.py:82] Create MLModel failed, status_code: 409, b'{"requestId":"4efe167c-e3f7-4fca-a996-514bd4331d8e","errorCode":"ItemDisplayNameAlreadyInUse","message":"Requested \'kmeans_segmentacion_clientes\' is already in use","isRetriable":false}'
Registered model 'kmeans_segmentacion_clientes' already exists. Creating a new version of this model...



🏁 CLUSTERING COMPLETADO


## 🔮 Inferencia y Validación del Modelo de Segmentación


In [19]:
print("📥 Recuperando el modelo desde MLflow...")

model_uri   = f"runs:/{run_id_clustering}/model"
loaded_kmeans = mlflow.sklearn.load_model(model_uri)
print("✅ Modelo K-Means cargado")

df_pandas["Cluster_Asignado"] = loaded_kmeans.predict(X_scaled)

print("\n📋 Vista de Producción: Clientes con Segmento Asignado")
print("=========================================================================")
display(df_pandas[["age", "price_per_unit", "quantity", "Cluster_Asignado"]].head(10))
print("=========================================================================")
print("\n🚀 Clustering finalizado y validado.")


StatementMeta(, 821ce642-c187-4906-9dde-32ed30228f9e, 21, Finished, Available, Finished, False)

📥 Recuperando el modelo desde MLflow...


✅ Modelo K-Means cargado

📋 Vista de Producción: Clientes con Segmento Asignado


SynapseWidget(Synapse.DataFrame, 8896815d-8d74-4464-a650-e89897afb7b6)


🚀 Clustering finalizado y validado.


StatementMeta(, 821ce642-c187-4906-9dde-32ed30228f9e, 29, Finished, Available, Finished, False)

# **CLASIFICACIÓN EXPERIMENTO**


## Documentación del Problema de ML

| Campo | Descripción |
|---|---|
| **Tipo de problema** | Clasificación binaria |
| **Variable objetivo** | `nivel_venta`: 1 si `total_amount >= 1000`, 0 en caso contrario |
| **Algoritmo** | Logistic Regression |
| **Métrica de éxito** | F1-Score ≥ 0.70 |

> ⚠️ **Nota sobre leakage**: `total_amount`, `quantity` y `price_per_unit` están directamente
> relacionados con el target. Se usan únicamente variables demográficas y de categoría.


In [20]:
import mlflow
import mlflow.sklearn
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

gold_table_lr      = "gold.dbo.gold_features_clasificacion"
experiment_name_lr = "ventas_high_value_classification"
model_name_lr      = "logistic_regression_ventas"

mlflow.set_experiment(experiment_name_lr)

print(f"📥 Dataset: {gold_table_lr}")
print(f"🔬 Experimento MLflow: {experiment_name_lr}")
print(f"📦 Modelo: {model_name_lr}")


StatementMeta(, 821ce642-c187-4906-9dde-32ed30228f9e, 22, Finished, Available, Finished, False)

📥 Dataset: gold.dbo.gold_features_clasificacion
🔬 Experimento MLflow: ventas_high_value_classification
📦 Modelo: logistic_regression_ventas


## 🏗️ Carga y EDA del Target


In [21]:
df_gold_lr = spark.table(gold_table_lr)

print(f"✓ Registros totales: {df_gold_lr.count():,}")
print(f"✓ Columnas         : {len(df_gold_lr.columns)}")

print("\nDistribución de nivel_venta (Target):")
df_gold_lr.groupBy("nivel_venta").count().orderBy("nivel_venta").show()

display(df_gold_lr.select("age", "grupo_edad", "id_genero", "quantity", "total_amount", "nivel_venta").limit(10))


StatementMeta(, 821ce642-c187-4906-9dde-32ed30228f9e, 23, Finished, Available, Finished, False)

✓ Registros totales: 1,000
✓ Columnas         : 11

Distribución de nivel_venta (Target):
+-----------+-----+
|nivel_venta|count|
+-----------+-----+
|          0|  798|
|          1|  202|
+-----------+-----+



SynapseWidget(Synapse.DataFrame, 5bb94b8a-576a-4a2b-9227-5a253e325f08)

## 📐 Preparación de Features (sin leakage)


In [22]:
from pyspark.sql.functions import col

feature_columns_lr = ["age", "quantity", "gender_F"]
label_column_lr    = "nivel_venta"

df_ml_lr = df_gold_lr.select(feature_columns_lr + [label_column_lr]).na.drop()

total_records_lr = df_ml_lr.count()

print(f"✅ Registros válidos: {total_records_lr:,}")
print(f"✅ Features         : {', '.join(feature_columns_lr)}")
print(f"✅ Label            : {label_column_lr}")

print("\n📊 Balance de clases:")
df_ml_lr.groupBy(label_column_lr).count().withColumn(
    "percentage", (col("count") / total_records_lr * 100)
).show()


StatementMeta(, 821ce642-c187-4906-9dde-32ed30228f9e, 24, Finished, Available, Finished, False)

✅ Registros válidos: 1,000
✅ Features         : age, quantity, gender_F
✅ Label            : nivel_venta

📊 Balance de clases:
+-----------+-----+------------------+
|nivel_venta|count|        percentage|
+-----------+-----+------------------+
|          1|  202|20.200000000000003|
|          0|  798| 79.80000000000001|
+-----------+-----+------------------+



## 🔀 Train/Test Split 80/20


In [23]:
train_df_lr, test_df_lr = df_ml_lr.randomSplit([0.8, 0.2], seed=42)

train_count_lr = train_df_lr.count()
test_count_lr  = test_df_lr.count()
total_count_lr = train_count_lr + test_count_lr

print(f"✅ Training set: {train_count_lr:,} registros ({train_count_lr / total_count_lr * 100:.1f}%)")
print(f"✅ Test set    : {test_count_lr:,}  registros ({test_count_lr  / total_count_lr * 100:.1f}%)")


StatementMeta(, 821ce642-c187-4906-9dde-32ed30228f9e, 25, Finished, Available, Finished, False)

✅ Training set: 838 registros (83.8%)
✅ Test set    : 162  registros (16.2%)


## 🔢 Vectorización con VectorAssembler


In [24]:
from pyspark.ml.feature import VectorAssembler

assembler_lr = VectorAssembler(inputCols=feature_columns_lr, outputCol="features")

train_assembled_lr = assembler_lr.transform(train_df_lr)
test_assembled_lr  = assembler_lr.transform(test_df_lr)

print("✅ Features vectorizadas")
train_assembled_lr.select("features", label_column_lr).show(5, truncate=False)


StatementMeta(, 821ce642-c187-4906-9dde-32ed30228f9e, 26, Finished, Available, Finished, False)

✅ Features vectorizadas
+--------------+-----------+
|features      |nivel_venta|
+--------------+-----------+
|[18.0,1.0,1.0]|0          |
|[18.0,1.0,1.0]|0          |
|[18.0,1.0,1.0]|0          |
|[18.0,1.0,1.0]|0          |
|[18.0,2.0,0.0]|0          |
+--------------+-----------+
only showing top 5 rows



## 🧠 Entrenamiento Logistic Regression y Tracking MLflow


In [25]:
import mlflow.sklearn
from sklearn.linear_model import LogisticRegression as SklearnLR
from mlflow.models import infer_signature

X_cols_lr = feature_columns_lr  # ya sin la etiqueta

print("🔄 Convirtiendo datos a pandas...")
train_pd_lr = train_df_lr.toPandas()
test_pd_lr  = test_df_lr.toPandas()

X_train_lr = train_pd_lr[X_cols_lr].values
y_train_lr = train_pd_lr[label_column_lr].values
X_test_lr  = test_pd_lr[X_cols_lr].values
y_test_lr  = test_pd_lr[label_column_lr].values

print(f"✅ Train: {len(X_train_lr):,} filas | Test: {len(X_test_lr):,} filas")

# Cerrar run previo
mlflow.end_run()

with mlflow.start_run(run_name="logistic_regression_ventas_v1") as run_lr:
    max_iter_lr = 100
    C_lr        = 100.0
    penalty_lr  = "l2"
    solver_lr   = "lbfgs"

    mlflow.log_param("max_iter",   max_iter_lr)
    mlflow.log_param("C",          C_lr)
    mlflow.log_param("penalty",    penalty_lr)
    mlflow.log_param("solver",     solver_lr)
    mlflow.log_param("features",   ", ".join(X_cols_lr))
    mlflow.log_param("train_size", len(X_train_lr))
    mlflow.log_param("test_size",  len(X_test_lr))

    lr_model = SklearnLR(
        max_iter=max_iter_lr, C=C_lr,
        penalty=penalty_lr, solver=solver_lr,
        random_state=42
    )

    print("🛠️ Entrenando Logistic Regression...")
    lr_model.fit(X_train_lr, y_train_lr)
    print("✅ Modelo entrenado exitosamente")

    # Predicciones y métricas
    y_pred_lr       = lr_model.predict(X_test_lr)
    y_pred_proba_lr = lr_model.predict_proba(X_test_lr)

    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

    accuracy_lr  = accuracy_score(y_test_lr, y_pred_lr)
    precision_lr = precision_score(y_test_lr, y_pred_lr, average="weighted")
    recall_lr    = recall_score(y_test_lr, y_pred_lr, average="weighted")
    f1_lr        = f1_score(y_test_lr, y_pred_lr, average="weighted")
    auc_lr       = roc_auc_score(y_test_lr, y_pred_proba_lr[:, 1])

    mlflow.log_metric("accuracy",  accuracy_lr)
    mlflow.log_metric("precision", precision_lr)
    mlflow.log_metric("recall",    recall_lr)
    mlflow.log_metric("f1_score",  f1_lr)
    mlflow.log_metric("roc_auc",   auc_lr)

    print(f"\n📏 Métricas de Clasificación")
    print("=" * 50)
    print(f"Accuracy              : {accuracy_lr:.4f}  ({accuracy_lr*100:.2f}%)")
    print(f"Precision (weighted)  : {precision_lr:.4f}  ({precision_lr*100:.2f}%)")
    print(f"Recall (weighted)     : {recall_lr:.4f}  ({recall_lr*100:.2f}%)")
    print(f"F1-Score              : {f1_lr:.4f}  ({f1_lr*100:.2f}%)")
    print(f"ROC-AUC               : {auc_lr:.4f}  ({auc_lr*100:.2f}%)")
    print("=" * 50)

    # Matriz de confusión como artefacto
    from sklearn.metrics import confusion_matrix
    import seaborn as sns
    cm_lr = confusion_matrix(y_test_lr, y_pred_lr)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_lr, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Venta Normal (0)", "Venta VIP (1)"],
                yticklabels=["Venta Normal (0)", "Venta VIP (1)"])
    plt.xlabel("Predicho", fontsize=12)
    plt.ylabel("Real",     fontsize=12)
    plt.title("Matriz de Confusión — Clasificación de Nivel de Venta", fontsize=14)
    plt.tight_layout()
    cm_path = "confusion_matrix_ventas.png"
    plt.savefig(cm_path)
    mlflow.log_artifact(cm_path)
    plt.close()
    print("\n✅ Matriz de confusión guardada como artefacto")

    # Signature y registro del modelo
    signature_lr = infer_signature(X_train_lr, y_pred_lr)
    mlflow.sklearn.log_model(lr_model, "model", signature=signature_lr)

    run_id_clasificacion = run_lr.info.run_id
    print("run_id_clasificacion")
    print(f"\n🏃 MLflow Run ID (Clasificación): {run_id_clasificacion}")

print("\n" + "="*60)
print("🏁 CLASIFICACIÓN COMPLETADA")
print("="*60)
print(f"F1-Score: {f1_lr:.4f}  |  ROC-AUC: {auc_lr:.4f}")
print(f"Experimento : {experiment_name_lr}")
print(f"Run ID      : {run_id_clasificacion}")


StatementMeta(, 821ce642-c187-4906-9dde-32ed30228f9e, 27, Finished, Available, Finished, False)

🔄 Convirtiendo datos a pandas...
✅ Train: 838 filas | Test: 162 filas

📏 Métricas de Clasificación
Accuracy              : 0.8272  (82.72%)
Precision (weighted)  : 0.6842  (68.42%)
Recall (weighted)     : 0.8272  (82.72%)
F1-Score              : 0.7489  (74.89%)
ROC-AUC               : 0.7499  (74.99%)

✅ Matriz de confusión guardada como artefacto
run_id_clasificacion

🏃 MLflow Run ID (Clasificación): ca86782f-6533-4a8a-a664-8a6dc5c16f3f


/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))



🏁 CLASIFICACIÓN COMPLETADA
F1-Score: 0.7489  |  ROC-AUC: 0.7499
Experimento : ventas_high_value_classification
Run ID      : ca86782f-6533-4a8a-a664-8a6dc5c16f3f


## 📊 Reporte de Clasificación Detallado


In [26]:
from sklearn.metrics import classification_report

print("📊 Reporte de Clasificación (Regresión Logística):")
print(classification_report(
    y_test_lr,
    y_pred_lr,
    target_names=["Venta Normal", "Venta VIP"]
))


StatementMeta(, 821ce642-c187-4906-9dde-32ed30228f9e, 28, Finished, Available, Finished, False)

/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.